# Session 3 — Control Flow, Functions, and Working with Files

Four parts today: making decisions and repeating work; packaging logic into functions; actually touching files and folders on disk; and a capstone that uses all of it to sort `sample_data/` into something usable. The capstone only needs what the core and stretch exercises cover across both sessions — nothing from a hard/homework tier is required to finish it.

In [ ]:
from pathlib import Path

data_dir = Path("sample_data")
if not data_dir.exists():
    raise FileNotFoundError(
        "sample_data/ not found — run setup_sample_data.py once, then re-run this cell."
    )

filenames = sorted(p.name for p in data_dir.iterdir())
print(f"Found {len(filenames)} files")

## Part 1 — if/else, loops, and combining conditions

`in` checks membership, whether a value shows up inside a list (or a string, or a few other things), and reads almost like English.

In [ ]:
known_types = ["appellate_body", "panel_report", "tariff_schedule", "legal_text", "ministerial_or_negotiation"]
doc_type = "appellate_body"

if doc_type in known_types:
    print("recognised category")
else:
    print("unrecognised category")

case_number = 316
if case_number > 300 and doc_type == "appellate_body":
    print("high case number, and an Appellate Body report")

`for` goes through every item in something once. Looping over a dictionary's `.items()`, previewed last session without actually looping, gives both the key and the value on each pass — unpacked directly into `key` and `value`, the same pattern as last session's tuple unpacking.

In [ ]:
document = {"filename": "WT-DS316-AB-R.txt", "case_number": 316, "doc_type": "appellate_body"}

for key, value in document.items():
    print(key, "->", value)

`break` stops a loop immediately, before it's gone through everything — useful the moment you only need the *first* match rather than every match.

In [ ]:
first_csv = None
for name in filenames:
    if name.lower().endswith(".csv"):
        first_csv = name
        break

print(first_csv)

`pass` does nothing at all — its entire purpose is to be a placeholder where Python's syntax requires *something* to follow a `:`, typically while a piece of logic is still being designed and one branch isn't written yet.

In [ ]:
for name in filenames:
    if name.lower().endswith(".csv"):
        pass  # tariff schedules: not handling these yet, come back to it later
    else:
        print(name)

Now the same ideas at folder scale: a `for` loop over every filename, an `if`/`elif`/`else` chain deciding what each one is. As before, a few files won't match anything, and that's expected, not a bug. The results go into a list, `classified`, rather than being thrown away, so later cells can build on this instead of redoing it.

In [ ]:
classified = []

for name in filenames:
    lower = name.lower()
    if "ab-r" in lower or lower.startswith("ab_") or lower.endswith("_ab.txt"):
        doc_type = "appellate_body"
    elif "panel" in lower or "-r_" in lower:
        doc_type = "panel_report"
    elif lower.endswith(".csv") or "tariff" in lower:
        doc_type = "tariff_schedule"
    elif "min" in lower or lower.startswith("tn_"):
        doc_type = "ministerial_or_negotiation"
    elif "agreement" in lower or "understanding" in lower or "gatt" in lower:
        doc_type = "legal_text"
    else:
        doc_type = "unclassified"
    classified.append((name, doc_type))

counts = {}
for name, doc_type in classified:
    counts[doc_type] = counts.get(doc_type, 0) + 1

for doc_type, n in counts.items():
    print(f"{doc_type}: {n}")

`while` repeats as long as a condition holds, rather than once per item, the right tool when you don't know the number of passes in advance. `continue` jumps straight to the next iteration without running the rest of the loop body, used here to set aside anything that isn't real work.

In [ ]:
queue = ["a.txt", "BROKEN_ENTRY", "b.txt", "c.txt", "BROKEN_ENTRY"]
processed = []
problems = []

while queue:
    current = queue.pop(0)
    if current == "BROKEN_ENTRY":
        problems.append(current)
        continue
    processed.append(current)

print("processed:", processed)
print("problems:", problems)

### Exercise 1

**Core.**
1. Loop through `filenames` and print each one alongside `True`/`False` for whether it ends in `.csv`.
2. Write an `if`/`elif`/`else` chain that classifies each filename as `"official-looking"` (starts with `WT-`, `G_`, or `TN_`) or `"renamed"`, a different question from the `doc_type` classification above; a file can be both an obvious Appellate Body report and clearly renamed by a person.
3. Define a list of at least five case numbers and use `in` to check whether `316` appears in it.
4. Using `break`, loop through `filenames` and stop as soon as you find the first one containing `"panel"`, printing which one it was.

**Stretch.**
1. Loop through your case-number list and print only the ones that are both greater than `200` and even (using `%`).
2. Loop through `filenames`, using `continue` to skip anything that doesn't end in `.txt` or `.csv`, printing only what passes.

**Hard (optional, homework).** Go through `classified` with a `while` loop and a manual index (not `for`), and, for each entry, use a nested `if` to also assign a rough confidence: `"high"` if the filename contains a recognisable dash-and-digits pattern like `"WT-"` or `"DS"` followed by digits (simple substring checks are fine, no regex needed), otherwise `"low"`. Stop as soon as you reach the first `"unclassified"` entry, printing its filename, doc_type, confidence, and how many entries you checked before reaching it.

In [ ]:
# Core
# your code here


# Stretch
# your code here


# Hard (optional)
# your code here


## Part 2 — Functions, in depth

The classification logic from Part 1, wrapped into a function, same logic, now callable by name on any single filename instead of only running over the whole folder at once.

In [ ]:
def classify_document(filename):
    lower = filename.lower()
    if "ab-r" in lower or lower.startswith("ab_") or lower.endswith("_ab.txt"):
        return "appellate_body"
    elif "panel" in lower or "-r_" in lower:
        return "panel_report"
    elif lower.endswith(".csv") or "tariff" in lower:
        return "tariff_schedule"
    elif "min" in lower or lower.startswith("tn_"):
        return "ministerial_or_negotiation"
    elif "agreement" in lower or "understanding" in lower or "gatt" in lower:
        return "legal_text"
    else:
        return "unclassified"

print(classify_document("WT-DS316-AB-R.txt"))
print(classify_document("notes_meeting_march.txt"))

Parameters can have default values, used whenever the caller doesn't supply that argument, handy for something like "the current year," usually the same but occasionally needing an override. A docstring, the string right after the `def` line, is what `help()` displays.

In [ ]:
def years_since(year, current_year=2024):
    """Return how many years have passed between `year` and `current_year`."""
    return current_year - year

print(years_since(1994))
print(years_since(1994, current_year=2030))
help(years_since)

Functions can call other functions, and can return more than one value at once as a tuple, unpacked directly into separate variables at the call site, the same pattern introduced last session.

In [ ]:
def guess_year(filename):
    for year in ["1994", "2017", "2019", "2021"]:
        if year in filename:
            return year
    return None

def describe_document(filename):
    doc_type = classify_document(filename)
    year = guess_year(filename)
    return doc_type, year

result_type, result_year = describe_document("AB_2019_7_footwear_import.txt")
print(result_type, result_year)

### Exercise 2

**Core.**
1. Write a function `is_csv(filename)` returning `True`/`False`.
2. Write `years_since(year, current_year=2024)` if you haven't already followed along above, and call it with and without overriding `current_year`.
3. Call both functions on three filenames from `sample_data` and print the results.

**Stretch.**
1. Write `summarize(filename)` that calls `classify_document` and `is_csv`, and returns both as a tuple; unpack the result into two named variables when you call it.
2. Add a docstring to one of your own functions and confirm it with `help(your_function)`.

**Hard (optional, homework).** Write `safe_guess_year(filename)` that calls `guess_year` internally, but returns the string `"unknown"` instead of `None` when nothing is found, using an `if` on `guess_year`'s return value inside your function. Small, but genuine practice composing one function's output as another's input.

In [ ]:
# Core
# your code here


# Stretch
# your code here


# Hard (optional)
# your code here


## Part 3 — pathlib, os, and shutil: working with real files

A `Path` carries more than just the string, `.exists()`, `.name`, `.suffix`, `.stem`, and `.parent` all answer questions you'd otherwise need string-slicing gymnastics for.

In [ ]:
p = Path("sample_data") / "WT-DS316-AB-R.txt"

print(p.exists())
print(p.name)     # WT-DS316-AB-R.txt
print(p.suffix)   # .txt
print(p.stem)     # WT-DS316-AB-R
print(p.parent)   # sample_data

Everything above uses `pathlib`, the modern, generally recommended way to work with paths. The older `os` and `os.path` modules do much the same job with a slightly different shape, and are common enough in other people's code, especially anything written more than a few years ago, that recognising them is worth a couple of minutes even though `pathlib` is what you'll reach for yourself.

In [ ]:
import os

print(os.getcwd())                                        # current working directory
print(os.listdir("sample_data")[:5])                       # same idea as data_dir.iterdir(), listed as plain strings
print(os.path.exists("sample_data/WT-DS316-AB-R.txt"))     # same as Path(...).exists()
print(os.path.join("sample_data", "WT-DS316-AB-R.txt"))    # same as Path("sample_data") / "WT-DS316-AB-R.txt"
print(os.path.basename("sample_data/WT-DS316-AB-R.txt"))   # same as Path(...).name

`.mkdir(parents=True, exist_ok=True)` creates a folder, and any missing parent folders, without complaining if it's already there. `shutil.copy` needs the destination folder to already exist, which is exactly what `.mkdir()` is for. `.rename()` moves or renames a file in place.

In [ ]:
import shutil

destination = Path("organized") / "appellate_body"
destination.mkdir(parents=True, exist_ok=True)

source = Path("sample_data") / "WT-DS316-AB-R.txt"
target = destination / source.name
shutil.copy(source, target)
print(target.exists())

renamed_target = destination / "renamed_example.txt"
target.rename(renamed_target)
print(renamed_target.exists(), target.exists())

Writing a file uses `open(path, "w")` as a context manager and `.write()`. Reading one back can be done a line at a time with `.readline()`, all at once as a list of lines with `.readlines()`, or, for the common case of just wanting the whole thing as one string, `Path.read_text()` is a shortcut for exactly that.

In [ ]:
report_path = Path("organized") / "report.txt"

with open(report_path, "w", encoding="utf-8") as f:
    f.write("Files organized so far: 2\n")
    f.write("Categories seen: appellate_body\n")

with open(report_path, "r", encoding="utf-8") as f:
    first_line = f.readline()
    print("first line only:", first_line)

with open(report_path, "r", encoding="utf-8") as f:
    all_lines = f.readlines()
    print("as a list of lines:", all_lines)

print("the pathlib shortcut:")
print(report_path.read_text())

### Exercise 3

**Core.**
1. Check whether `sample_data/does_not_exist.txt` exists (should be `False`) and `sample_data/WT-DS316-AB-R.txt` does (should be `True`), using `pathlib`.
2. For three filenames of your choice, print `.suffix` and `.stem` separately.
3. Create a folder called `my_test_folder` with `.mkdir(exist_ok=True)` and confirm it exists.

**Stretch.**
1. Copy two files of your choice into `my_test_folder` with `shutil.copy`, then confirm both are there using `.rglob("*")`.
2. Write a text file into `my_test_folder` listing the names of the files you copied, one per line, then read it back with `.readlines()` and print the result.

**Hard (optional, homework).** Write `safe_copy(source_path, destination_folder)` that copies a file into a destination folder, but first checks with `.exists()` whether a file of that name is already there. If so, instead of overwriting it, copy it under a modified name (appending `_copy` to the stem, keeping the original suffix) and return the path actually used.

In [ ]:
# Core
# your code here


# Stretch
# your code here


# Hard (optional)
# your code here


## Capstone — organizing the folder, with a report

Everything from both sessions comes together here, using only what the core and stretch exercises already cover.

Steps:

1. Get every file in `sample_data/` using `pathlib`.
2. For each one, call `describe_document` (Exercise 2) to get its `doc_type` and `year`.
3. Build a destination path `organized/<doc_type>/<year>/`, using `"unknown_year"` in place of `<year>` when it's `None`.
4. Create that folder if needed, and copy the file into it.
5. Once every file is handled, write a short summary to `organized/report.txt`, counts per `doc_type`, and how many files had no recognisable year, and print the same summary to the screen.

The classification and year-guessing rules here are simplified for teaching purposes and would need broadening for real use on an actual downloads folder, but the shape of the script, and everything it's built from, would not change.

In [ ]:
organized_dir = Path("organized")

# your code here



---

That's both sessions. Worked solutions for every exercise, at every tier, are in the accompanying solutions notebook, worth comparing against after attempting these rather than before. Beyond the WTO framing, the pattern built up over these two sessions, loop over a folder, decide something about each item, act on it, write down what happened, covers a large share of what actually eats time in research work that has nothing to do with trade law specifically.